# cracksteam456 Voice Studio

Phát triển và duy trì bởi [cracksteam456](https://github.com/cracksteam456/voice-studio).

150 giọng hot · lọc nhóm/ngôn ngữ · tải giọng mẫu riêng · TXT · WAV/MP3 · ZIP từng đoạn.

1. Chọn **Runtime → Change runtime type → T4 GPU**.
Nếu chỉ có CPU, notebook vẫn chạy nhưng tạo giọng sẽ chậm hơn.
2. Chạy ô cài đặt, rồi chọn **Runtime → Restart session**.
3. Chạy ô khởi động model, rồi ô giao diện cuối cùng.
Bộ 150 giọng tải từ GitHub Release của cracksteam456: đặt GITHUB_TOKEN trong Colab Secrets (quyền đọc repo private), hoặc chọn ZIP từ máy.
4. Nhập nội dung, chọn giọng, bấm **Tạo giọng nói** và tải kết quả.

Mỗi phiên chỉ tải model một lần. Khi đổi giọng hoặc tốc độ, dùng giao diện, không chạy lại ô model.
Giữ Colab hoạt động khi dùng link giao diện. Các file âm thanh là file tạm; tải về trước khi ngắt phiên.


In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'omnivoice==0.2.1', 'gradio', 'numpy==2.1.3', 'scipy==1.15.3', 'requests==2.32.4'], check=True)

# Verify native imports in a fresh process; the notebook kernel may still hold old modules.
subprocess.run([sys.executable, '-c', 'import numpy, scipy.linalg; print("NumPy/SciPy: OK")'], check=True)


print('Đã cài thư viện. Chọn Runtime → Restart session, rồi chạy ô khởi động model.')

print('Đã cài thư viện. Chọn Runtime → Restart session, rồi chạy ô khởi động model.')

print('Đã cài thư viện. Chọn Runtime → Restart session, rồi chạy ô khởi động model.')


## Khởi động giọng đọc

Sau ô cài đặt, khởi động lại phiên rồi chạy ô này. Ô này kiểm tra NumPy/SciPy, nạp model và mở giao diện tạo WAV.

In [ ]:
"""Load OmniVoice on the accelerator actually available in this runtime."""
from importlib.metadata import version
from types import SimpleNamespace
import sys

for package in ('numpy', 'scipy'):
    loaded = sys.modules.get(package)
    if loaded is not None and loaded.__version__ != version(package):
        raise RuntimeError('Chọn Runtime → Restart session, rồi chạy lại ô khởi động (không cần cài lại).')

import numpy as np
import scipy.linalg
import torch
import transformers as _tf

if not hasattr(torch, '_utils') and hasattr(torch._C, '_utils'):
    torch._utils = torch._C._utils

if not hasattr(_tf, 'AutoFeatureExtractor'):
    class _SafeAutoFeatureExtractor:
        @staticmethod
        def from_pretrained(model_name, **kwargs):
            from transformers import AutoConfig
            config = AutoConfig.from_pretrained(model_name, **kwargs)
            return SimpleNamespace(sampling_rate=getattr(config, 'sampling_rate', 24000))
    _tf.AutoFeatureExtractor = _SafeAutoFeatureExtractor

from omnivoice import OmniVoice, OmniVoiceGenerationConfig

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if torch.cuda.is_available() else torch.float32
if device == 'cpu':
    torch.set_num_threads(min(4, torch.get_num_threads()))
    print('Đang dùng CPU: tạo giọng chậm hơn GPU. Có thể chọn T4 trong Runtime → Change runtime type.', flush=True)
else:
    print('GPU:', torch.cuda.get_device_name(0), flush=True)

if 'model' not in globals() or str(model.device) != device:
    print('Đang tải OmniVoice…', flush=True)
    model = OmniVoice.from_pretrained('k2-fsa/OmniVoice', device_map=device,
                                      dtype=dtype, load_asr=False)
print('MODEL_READY', device, '—', model.sampling_rate, 'Hz', flush=True)


## Mở Voice Studio
Chạy sau khi ô model hoàn tất. Có thể chạy lại ô này để mở lại giao diện mà không tải lại model.

In [ ]:
if 'demo' in globals():
    demo.close()

"""Load the project's 150-voice pack locally or from its private GitHub release."""
from pathlib import Path
from urllib.request import Request, build_opener, HTTPRedirectHandler
from urllib.parse import urlsplit
from urllib.error import HTTPError
import hashlib
import json
import os
import tempfile
import zipfile

PACK_RELEASE = 'https://api.github.com/repos/cracksteam456/voice-studio/releases/tags/hot-voices-2026-11-17'
PACK_NAME = 'hot-voices-150.zip'
GROUP_NAMES = {'vietnamese_primary_50': 'Tiếng Việt · Top 50',
               'trending_global_50': 'Trending toàn cầu · Top 50',
               'usage_global_50': 'Dùng nhiều toàn cầu · Top 50'}


def read_voice_pack(root):
    root = Path(root).resolve()
    entries = json.loads((root / 'catalog.json').read_text(encoding='utf-8'))
    if not isinstance(entries, list) or len(entries) != 150:
        raise ValueError('Bộ giọng phải có đủ 150 bản ghi.')
    seen = set()
    for entry in entries:
        relative = entry.get('sample_file', '')
        path = (root / relative).resolve()
        if (not relative or not path.is_relative_to(root) or not path.is_file()
                or path.suffix.lower() not in ('.wav', '.mp3', '.m4a', '.flac', '.ogg')):
            raise ValueError('Đường dẫn mẫu trong catalog không hợp lệ.')
        if relative in seen or entry.get('group') not in GROUP_NAMES:
            raise ValueError('Catalog có mẫu trùng hoặc nhóm không hợp lệ.')
        seen.add(relative)
        expected = entry.get('sample_sha256')
        if not expected or hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise ValueError('File mẫu thiếu hoặc không khớp checksum: ' + relative)
    return entries


def unpack_voice_pack(archive, destination):
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Validate in a fresh folder; a failed upload never damages the current pack.
    with tempfile.TemporaryDirectory(dir=destination.parent) as work:
        root = Path(work).resolve()
        with zipfile.ZipFile(archive) as bundle:
            entries = bundle.infolist()
            if len(entries) > 200 or sum(e.file_size for e in entries) > 60_000_000:
                raise ValueError('Gói mẫu vượt giới hạn kích thước.')
            for entry in entries:
                target = (root / entry.filename).resolve()
                if ('\\' in entry.filename or not target.is_relative_to(root)
                        or entry.file_size > 25_000_000 or (entry.external_attr >> 16) & 0o170000 == 0o120000):
                    raise ValueError('Gói mẫu chứa đường dẫn không hợp lệ.')
            bundle.extractall(root)
        read_voice_pack(root)
        destination.mkdir(parents=True, exist_ok=True)
        import shutil
        for source in root.rglob('*'):
            if source.is_file():
                target = destination / source.relative_to(root)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(source, target)
    return destination


def download_voice_pack(token, destination):
    # Send credentials only to api.github.com, never to a release download redirect.
    class NoRedirect(HTTPRedirectHandler):
        def redirect_request(self, *args, **kwargs):
            return None
    opener = build_opener(NoRedirect())
    headers = {'Accept': 'application/vnd.github+json', 'User-Agent': 'cracksteam456-voice-studio'}
    if token:
        headers['Authorization'] = 'Bearer ' + token
    with opener.open(Request(PACK_RELEASE, headers=headers), timeout=30) as response:
        release = json.loads(response.read(1_000_000))
    asset = next(a for a in release['assets'] if a['name'] == PACK_NAME)
    url = asset['url']
    if urlsplit(url).hostname != 'api.github.com' or not url.startswith('https://api.github.com/'):
        raise ValueError('Địa chỉ release không hợp lệ.')
    headers['Accept'] = 'application/octet-stream'
    try:
        response = opener.open(Request(url, headers=headers), timeout=45)
    except HTTPError as error:
        if error.code not in (301, 302, 303, 307, 308):
            raise
        url = error.headers['Location']
        parsed = urlsplit(url)
        if parsed.scheme != 'https' or parsed.hostname not in ('release-assets.githubusercontent.com', 'objects.githubusercontent.com'):
            raise ValueError('Địa chỉ tải release không hợp lệ.') from None
        response = opener.open(Request(url, headers={'User-Agent': headers['User-Agent']}), timeout=45)
    with response, tempfile.TemporaryDirectory() as temp:
        path = Path(temp) / PACK_NAME
        with path.open('wb') as output:
            total = 0
            while data := response.read(256 * 1024):
                total += len(data)
                if total > 40_000_000:
                    raise ValueError('Gói tải vượt 40 MB.')
                output.write(data)
        return unpack_voice_pack(path, destination)


def load_voice_pack():
    for root in (Path('outputs/hot-voices'), Path('/content/hot-voices')):
        if (root / 'catalog.json').exists():
            return root.resolve(), read_voice_pack(root)
    target = Path('/content/hot-voices') if Path('/content').is_dir() else Path('outputs/hot-voices')
    if Path(PACK_NAME).exists():
        root = unpack_voice_pack(PACK_NAME, target)
        return root, read_voice_pack(root)
    token = os.environ.get('GITHUB_TOKEN', '')
    if not token and Path('/content').is_dir():
        try:
            from google.colab import userdata
            token = userdata.get('GITHUB_TOKEN')
        except Exception:
            pass
    if not token:
        from getpass import getpass
        print('Repo private: dùng GitHub token quyền Contents: Read cho cracksteam456/voice-studio. '
              'Để trống để tải ZIP từ máy lên Colab. Token không được lưu.', flush=True)
        token = getpass('GitHub token (ẩn, hoặc Enter để chọn ZIP): ').strip()
    if token:
        try:
            root = download_voice_pack(token, target)
        except HTTPError as error:
            raise RuntimeError(f'GitHub HTTP {error.code}: kiểm tra token có quyền đọc repo private.') from None
    else:
        from google.colab import files
        print('Chọn hot-voices-150.zip (24 MB) đã tải từ GitHub Release.', flush=True)
        uploaded = files.upload()
        if PACK_NAME not in uploaded:
            raise ValueError('Cần chọn đúng file ' + PACK_NAME)
        root = unpack_voice_pack(PACK_NAME, target)
    return root, read_voice_pack(root)


"""Colab UI; run after loading `model` and `OmniVoiceGenerationConfig`."""
from collections import OrderedDict
from pathlib import Path
import json
import re
import subprocess
import tempfile
import time
import zipfile

import gradio as gr
import numpy as np
import soundfile as sf
import torch

if 'load_voice_pack' not in globals():
    from voice_assets import load_voice_pack, GROUP_NAMES
SAMPLE_ROOT, CATALOG = load_voice_pack()
VOICES = {entry['voice_id']: entry for entry in CATALOG}
DEFAULT_GROUP = 'Tiếng Việt · Top 50'
LANGUAGES = {'Tiếng Việt': 'vi', 'English': 'en', '日本語': 'ja', '한국어': 'ko',
             '中文': 'zh', 'Français': 'fr', 'Deutsch': 'de', 'Español': 'es'}
PROMPTS = OrderedDict()
SR = model.sampling_rate


def sample_path(voice_id):
    if voice_id not in VOICES:
        raise gr.Error('Chọn một giọng trong danh sách.')
    return str(SAMPLE_ROOT / VOICES[voice_id]['sample_file'])


def voice_choices(group, language):
    return [(f"{v['rank']:02d} · {v['name']} [{v.get('primary_language') or '?'}]", v['voice_id'])
            for v in CATALOG if (group == 'Tất cả' or GROUP_NAMES[v['group']] == group)
            and (language == 'Tất cả' or v.get('primary_language') == language)]


def filter_voices(group, language):
    choices = voice_choices(group, language)
    selected = choices[0][1] if choices else None
    return gr.Dropdown(choices=choices, value=selected), sample_path(selected) if selected else None


def import_text(path):
    if not path:
        raise gr.Error('Chọn file TXT trước.')
    path = Path(path)
    if path.stat().st_size > 1_000_000:
        raise gr.Error('File TXT quá lớn (tối đa 1 MB).')
    try:
        return path.read_text(encoding='utf-8-sig')
    except UnicodeError:
        raise gr.Error('Lưu file TXT bằng UTF-8 rồi thử lại.') from None


def split_text(text, limit):
    """Preserve every word, splitting at blank lines, sentences, then spaces."""
    chunks = []
    for paragraph in re.split(r'\n\s*\n', text.strip()):
        words = paragraph.split()
        current = ''
        for word in words:
            if len(word) > limit:
                raise ValueError('Có một từ quá dài. Hãy thêm khoảng trắng vào văn bản.')
            if current and len(current) + len(word) + 1 > limit:
                chunks.append(current)
                current = ''
            current = (current + ' ' + word).strip()
            if len(current) >= limit * 0.6 and re.search(r'[.!?…。！？][\"\u201d\u2019]?$' , word):
                chunks.append(current)
                current = ''
        if current:
            chunks.append(current)
    return chunks


def get_prompt(name, custom, reference_text, progress=gr.Progress()):
    reference_text = reference_text or ''
    path = custom if custom else sample_path(name)
    if not Path(path).is_file():
        raise gr.Error('Không tìm thấy file giọng mẫu.')
    if Path(path).stat().st_size > 25_000_000:
        raise gr.Error('Giọng mẫu tối đa 25 MB; nên dùng đoạn sạch dài 3–10 giây.')
    key = (str(path), Path(path).stat().st_mtime_ns, reference_text.strip())
    if key not in PROMPTS:
        kwargs = {'ref_audio': str(path)}
        if reference_text.strip():
            kwargs['ref_text'] = reference_text.strip()
        elif not getattr(model, '_asr_pipe', None):
            progress(0.02, desc='Tải Whisper để nhận dạng lời trong mẫu (chỉ lần đầu)…')
            model.load_asr_model()
        progress(0.06, desc='Mã hóa mẫu giọng (lần đầu cho giọng này)…')
        PROMPTS[key] = model.create_voice_clone_prompt(**kwargs)
        if len(PROMPTS) > 8:
            PROMPTS.popitem(last=False)
    PROMPTS.move_to_end(key)
    return PROMPTS[key]


def synthesize(text, name, custom, reference_text, language, speed, pause,
               steps, guidance, denoise, seed, chunk_size, mp3, progress=gr.Progress()):
    text = (text or '').strip()
    if not text:
        raise gr.Error('Nhập văn bản hoặc nạp file TXT trước.')
    if len(text) > 100_000:
        raise gr.Error('Tối đa 100.000 ký tự mỗi lần; hãy chia kịch bản thành nhiều lượt.')
    try:
        chunks = split_text(text, int(chunk_size))
    except ValueError as error:
        raise gr.Error(str(error)) from error
    if language not in LANGUAGES:
        raise gr.Error('Chọn ngôn ngữ trong danh sách.')
    progress(0, desc='Chuẩn bị giọng mẫu…')
    started = time.monotonic()
    try:
        device = 'GPU' if torch.cuda.is_available() else 'CPU'
        status = f'Khởi tạo mẫu giọng trên {device}…'
        prompt = get_prompt(name, custom, reference_text, progress)
        prompt_seconds = time.monotonic() - started
        config = OmniVoiceGenerationConfig(
            num_step=int(steps), guidance_scale=float(guidance), denoise=bool(denoise),
            preprocess_prompt=True, postprocess_output=True,
            position_temperature=5.0, class_temperature=0.2,
            pad_duration=0.1, fade_duration=0.1,
        )
        output = Path(tempfile.mkdtemp(prefix='voice-studio-'))
        audio_parts, manifest, segment_files = [], [], []
        cursor = 0.0
        with torch.random.fork_rng():
            if int(seed) >= 0:
                torch.manual_seed(int(seed))
            for index, chunk in enumerate(chunks):
                status = f'Đang đọc đoạn {index + 1}/{len(chunks)} trên {device}…'
                progress(0.08 + index / len(chunks) * 0.88, desc=status)
                audio = np.asarray(model.generate(
                    text=chunk, voice_clone_prompt=prompt, language=LANGUAGES[language],
                    speed=float(speed), generation_config=config,
                )[0], dtype=np.float32).reshape(-1)
                if not audio.size or not np.isfinite(audio).all():
                    raise RuntimeError(f'Âm thanh đoạn {index + 1} không hợp lệ.')
                audio = np.clip(audio, -1, 1)
                generation_seconds = time.monotonic() - started - prompt_seconds
                segment = output / f'doan-{index + 1:03d}.wav'
                sf.write(segment, audio, SR, subtype='PCM_16')
                segment_files.append(segment)
                duration = len(audio) / SR
                manifest.append({'segment': index + 1, 'text': chunk, 'start': cursor, 'end': cursor + duration})
                audio_parts.append(audio)
                cursor += duration
                if index < len(chunks) - 1:
                    silence = np.zeros(round(SR * float(pause)), dtype=np.float32)
                    audio_parts.append(silence)
                    cursor += len(silence) / SR
        wav = output / 'giong-doc.wav'
        sf.write(wav, np.concatenate(audio_parts), SR, subtype='PCM_16')
        (output / 'kich-ban.txt').write_text(text, encoding='utf-8')
        (output / 'cac-doan.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
        settings = dict(voice='Giọng riêng' if custom else VOICES[name]['name'],
                        voice_id=None if custom else name, language=language, speed=speed,
                        pause=pause, steps=steps, guidance=guidance, denoise=denoise,
                        seed=seed, chunk_size=chunk_size, sample_rate=SR)
        (output / 'thiet-lap.json').write_text(json.dumps(settings, ensure_ascii=False, indent=2), encoding='utf-8')
        downloads = [str(wav)]
        export_notice = ''
        if mp3:
            mp3_path = output / 'giong-doc.mp3'
            try:
                subprocess.run(['ffmpeg', '-y', '-v', 'error', '-i', str(wav), '-b:a', '192k', str(mp3_path)],
                               check=True, capture_output=True, timeout=120)
                downloads.append(str(mp3_path))
            except (OSError, subprocess.SubprocessError):
                export_notice = ' Chưa xuất được MP3; WAV và ZIP vẫn dùng được.'
        archive = output / 'goi-giong-doc.zip'
        with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as bundle:
            for path in segment_files + [output / 'kich-ban.txt', output / 'cac-doan.json', output / 'thiet-lap.json']:
                bundle.write(path, path.name)
        downloads.append(str(archive))
        progress(1, desc='Hoàn tất')
        return str(wav), downloads, (f'Hoàn tất trên {device} • audio {cursor:.1f} giây • '
            f'{prompt_seconds:.1f} giây chuẩn bị mẫu • {generation_seconds:.1f} giây tạo giọng • '
            f'{len(chunks)} đoạn.{export_notice}')
    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        raise gr.Error('GPU hết bộ nhớ. Giảm độ dài mỗi đoạn hoặc khởi động lại phiên.') from None


with gr.Blocks(title='cracksteam456 Voice Studio') as demo:
    gr.Markdown('# cracksteam456 Voice Studio\n150 giọng hot · 50 tiếng Việt · 50 Trending · 50 được dùng nhiều')
    gr.Textbox(label='Thiết bị xử lý', interactive=False,
               value=torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU · không có GPU · tạo giọng có thể mất vài phút mỗi câu')
    with gr.Row():
        with gr.Column(scale=3, min_width=280):
            script = gr.Textbox(label='Kịch bản', lines=12, placeholder='Dán nội dung cần đọc. Dùng dòng trống để tách đoạn.')
            with gr.Row():
                text_file = gr.File(label='Nhập kịch bản TXT (UTF-8)', file_types=['.txt'], type='filepath')
                load_text = gr.Button('Nạp văn bản')
            with gr.Row():
                create = gr.Button('Tạo giọng nói', variant='primary')
                clear = gr.ClearButton([script], value='Xóa văn bản')
            status = gr.Textbox(label='Trạng thái', value='Sẵn sàng. Chọn giọng, nhập văn bản rồi bấm tạo.', interactive=False)
            result = gr.Audio(label='Nghe kết quả', type='filepath', format='wav')
            files = gr.File(label='Tải WAV / MP3 / ZIP từng đoạn', file_count='multiple', interactive=False)
        with gr.Column(scale=2, min_width=280):
            with gr.Row():
                group = gr.Dropdown(['Tất cả', *GROUP_NAMES.values()], value=DEFAULT_GROUP, label='Nhóm giọng')
                voice_language = gr.Dropdown(['Tất cả', *sorted({v.get('primary_language') for v in CATALOG if v.get('primary_language')})],
                                             value='Tất cả', label='Ngôn ngữ mẫu')
            initial_choices = voice_choices(DEFAULT_GROUP, 'Tất cả')
            voice = gr.Dropdown(initial_choices, value=initial_choices[0][1], label='Giọng trong bộ 150 mẫu', filterable=True)
            preview = gr.Button('Nghe giọng mẫu')
            sample = gr.Audio(label='Giọng mẫu có sẵn', interactive=False)
            with gr.Accordion('Dùng giọng mẫu riêng', open=False):
                gr.Markdown('Tải đoạn giọng sạch dài khoảng 3–10 giây. Khi có file, giọng riêng sẽ được ưu tiên. Xóa file để quay lại giọng có sẵn.')
                custom = gr.Audio(label='Giọng mẫu của bạn', sources=['upload'], type='filepath')
                ref_text = gr.Textbox(label='Lời nói trong mẫu (không bắt buộc)', lines=2,
                                      placeholder='Để trống để tự nhận dạng; điền đúng lời giúp giảm thời gian chuẩn bị.')
            lang = gr.Dropdown(list(LANGUAGES), value='Tiếng Việt', label='Ngôn ngữ đọc')
            speed = gr.Slider(0.6, 1.5, value=0.95, step=0.05, label='Tốc độ đọc (1 = bình thường)')
            pause = gr.Slider(0, 2, value=0.3, step=0.1, label='Khoảng nghỉ giữa các đoạn (giây)')
            mp3 = gr.Checkbox(value=True, label='Xuất thêm MP3')
            with gr.Accordion('Chất lượng và văn bản dài', open=False):
                steps = gr.Radio([16, 32, 64], value=32 if torch.cuda.is_available() else 16, label='Số bước (16 nhanh · 32 cân bằng · 64 chậm hơn)')
                guidance = gr.Slider(1, 4, value=1.8, step=0.1, label='Guidance — mức bám giọng mẫu')
                denoise = gr.Checkbox(value=True, label='Giảm nhiễu giọng đọc')
                chunk_size = gr.Slider(150, 1000, value=400, step=50, label='Độ dài tối đa mỗi đoạn (ký tự)')
                seed = gr.Number(value=-1, precision=0, label='Seed (-1 = ngẫu nhiên; cùng seed giúp tái tạo)')
    gr.Markdown('ZIP gồm WAV từng đoạn, kịch bản, thiết lập và mốc thời gian từng đoạn (không phải phụ đề theo từ). '
                'File tạo ra nằm tạm trên Colab; tải về trước khi phiên kết thúc.')
    preview.click(sample_path, voice, sample, queue=False)
    group.change(filter_voices, [group, voice_language], [voice, sample], queue=False)
    voice_language.change(filter_voices, [group, voice_language], [voice, sample], queue=False)
    voice.change(lambda selected: sample_path(selected) if selected else None, voice, sample, queue=False)
    load_text.click(import_text, text_file, script, queue=False)
    create.click(synthesize, [script, voice, custom, ref_text, lang, speed, pause, steps, guidance,
                             denoise, seed, chunk_size, mp3], [result, files, status], concurrency_limit=1)

demo.queue(default_concurrency_limit=1, max_size=10)
demo.launch(share=True, debug=False)
